<a href="https://colab.research.google.com/github/Euler912/Thesis/blob/main/PyTorch-MDCF-Optimization-Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
from torch.autograd.functional import hessian
import cvxpy as cp
import numpy as np
import time
import scipy.linalg
import pandas as pd


# 1. MDCF SOLVER CLASS

In [2]:

# Ensure device is set
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on device: {device}")

class MDCF:
    @staticmethod
    def get_clean_gradient(func, x_tensor):
        """Calculates gradient and returns it as a NUMPY array for CVXPY."""
        x = x_tensor.detach().clone().requires_grad_(True)
        y = func(x)
        y.backward()
        return x.grad.detach().cpu().numpy()

    @staticmethod
    def get_hessian(func, x_tensor):
        """Calculates Hessian using PyTorch's built-in functional API."""
        return torch.autograd.functional.hessian(func, x_tensor).to(device)

    @staticmethod
    def solve_initialization(g_cvxpy, constraints, n):
        """Finds x_g by minimizing g(x)."""
        x = cp.Variable(n)
        prob = cp.Problem(cp.Minimize(g_cvxpy(x)), constraints)

        # Try CLARABEL first, fall back to SCS
        try:
            prob.solve(solver=cp.CLARABEL, verbose=False)
        except:
            prob.solve(solver=cp.SCS, verbose=False)

        if x.value is None:
            # If optimization failed, return midpoint of feasible region
            print("Warning: Initialization failed, using midpoint of constraints")
            lower_bounds = (1 + np.arange(1, n + 1)) / 90.0
            upper_bounds = (1 + 5 * np.arange(1, n + 1)) / 90.0
            x_init = (lower_bounds + upper_bounds) / 2.0
            return torch.tensor(x_init, dtype=torch.float64)

        return torch.tensor(x.value, dtype=torch.float64)

    @staticmethod
    def compute_analytic_center(A_list, b_list, n, solver_opts):
        """Computes the analytic center x_ac."""
        x = cp.Variable(n)
        log_barrier_terms = []
        for A, b in zip(A_list, b_list):
            if A.ndim == 1:
                log_barrier_terms.append(cp.log(b - A @ x))
            else:
                log_barrier_terms.append(cp.sum(cp.log(b - A @ x)))

        prob = cp.Problem(cp.Maximize(cp.sum(log_barrier_terms)))
        try:
            prob.solve(**solver_opts)
        except:
            prob.solve(solver=cp.SCS, eps=1e-6)

        if x.value is None:
            print("Warning: Analytic center computation failed, using feasible point")
            lower_bounds = (1 + np.arange(1, n + 1)) / 90.0
            upper_bounds = (1 + 5 * np.arange(1, n + 1)) / 90.0
            return (lower_bounds + upper_bounds) / 2.0

        return x.value

    @staticmethod
    def barrier_hessian(x_np, A_list, b_list):
        """Computes Hessian of the log-barrier function."""
        n = len(x_np)
        H = np.zeros((n, n))
        for A, b in zip(A_list, b_list):
            if A.ndim == 1:
                denom = (b - np.dot(A, x_np))**2
                if denom > 1e-10:
                    H += np.outer(A, A) / denom
            else:
                s = b - A @ x_np
                # Avoid division by very small numbers
                s = np.maximum(s, 1e-10)
                scaled_A = A / s[:, None]
                H += scaled_A.T @ scaled_A

        # Ensure positive definiteness
        H = H + np.eye(n) * 1e-6
        return H

    @staticmethod
    def hidden_convex_fast(D_np, x1_np, e_np, Q_np, q_np):
        """Solves the hidden convex subproblem: Maximize Quadratic over Ellipsoid."""
        Q_np = Q_np + np.eye(len(Q_np)) * 1e-6
        D = torch.tensor(D_np, device=device, dtype=torch.float64)
        Q = torch.tensor(Q_np, device=device, dtype=torch.float64)
        x1 = torch.tensor(x1_np, device=device, dtype=torch.float64).view(-1, 1)
        e = torch.tensor(e_np, device=device, dtype=torch.float64).view(-1, 1)
        q = torch.tensor(q_np, device=device, dtype=torch.float64).view(-1, 1)

        try:
            L = torch.linalg.cholesky(Q)
            L_inv = torch.linalg.inv(L)
            A_mat = L_inv @ D @ L_inv.T
            alfa, U = torch.linalg.eigh(A_mat)
            S = L_inv.T @ U
            beta = S.T @ (D @ x1 - e)

            alfa_np = alfa.detach().cpu().numpy().flatten()
            beta_np = beta.detach().cpu().numpy().flatten()
            delta_val = (-2 * S.T @ Q @ q).detach().cpu().numpy().flatten()
            gamma_val = float((q.T @ Q @ q).detach().item())

            y = cp.Variable(len(alfa_np))
            z = cp.Variable(len(alfa_np))

            constraints = [
                2 * cp.sum(y) + delta_val @ z + gamma_val <= 1,
                cp.square(z) <= 2 * y
            ]
            prob = cp.Problem(cp.Minimize(-alfa_np @ y + beta_np @ z), constraints)

            solver_to_use = cp.CLARABEL if 'CLARABEL' in cp.installed_solvers() else cp.SCS
            prob.solve(solver=solver_to_use, verbose=False)

            if z.value is None:
                return x1_np
            z_res = torch.tensor(z.value, device=device, dtype=torch.float64).view(-1, 1)
            x_final = (S @ z_res).view(-1).detach().cpu().numpy()
            return x_final
        except:
            return x1_np

    @staticmethod
    def solve(f_torch, g_torch, f_cp, g_cp, A_list, b_list, n):
        SOLVER_OPTS = {'solver': cp.CLARABEL, 'tol_gap_abs': 1e-9, 'tol_gap_rel': 1e-9, 'verbose': False} \
                      if 'CLARABEL' in cp.installed_solvers() else {'solver': cp.SCS, 'eps': 1e-9, 'verbose': False}

        def get_constraints(var):
            return [A @ var <= b for A, b in zip(A_list, b_list)]

        # 1. Initialization
        print("Step 1: Initialization...")
        x_g_var = cp.Variable(n)
        x_init = MDCF.solve_initialization(g_cp, get_constraints(x_g_var), n)

        # 2. Analytic Center
        print("Step 2: Computing analytic center...")
        x_ac = MDCF.compute_analytic_center(A_list, b_list, n, SOLVER_OPTS)
        print(f"  Analytic center computed. Mean = {np.mean(x_ac):.6f}")

        Q_barrier = MDCF.barrier_hessian(x_ac, A_list, b_list)

        # 3. Quadratic Approximation of Objective (at Initial point)
        print("Step 3: Computing Hessian and gradient...")
        x_init_tensor = torch.tensor(x_ac, device=device, dtype=torch.float64)
        D_f = MDCF.get_hessian(f_torch, x_init_tensor).cpu().numpy()
        grad_f = MDCF.get_clean_gradient(f_torch, x_init_tensor)

        # 4. Hidden Convexity (The Global Jump)
        print("Step 4: Hidden convexity jump (x_in)...")
        x_in = MDCF.hidden_convex_fast(D_f, x_ac, grad_f, Q_barrier, x_ac)

        m_constraints = sum(len(b) for b in b_list)
        scale_factor = (m_constraints * (1 + 2/np.sqrt(m_constraints)))**2
        Q_out = Q_barrier / scale_factor

        print("Step 4b: Hidden convexity jump (x_out)...")
        x_out = MDCF.hidden_convex_fast(D_f, x_ac, grad_f, Q_out, x_ac)

        # 5. Line Search
        print("Step 5: Line search...")
        x_mid = x_in
        for alpha in np.linspace(0, 1, 20):
            candidate = alpha * x_out + (1 - alpha) * x_in
            feasible = True
            for A, b in zip(A_list, b_list):
                if np.any(A @ candidate > b + 1e-5):
                    feasible = False
                    break
            if feasible:
                x_mid = candidate
                break

        # 6. DCA Refinement
        print("Step 6: DCA refinement...")
        candidates = [x_mid, x_in, x_ac]
        best_val = -np.inf
        best_x = None

        for idx, start_point in enumerate(candidates):
            print(f"  Refining from candidate {idx+1}/3...")
            x_curr = start_point
            for k in range(50):
                x_prev = x_curr
                # Linearize the CONVEX part we want to maximize (gradient ascent)
                x_curr_tensor = torch.tensor(x_curr, device=device, dtype=torch.float64)
                grad_f_np = MDCF.get_clean_gradient(f_torch, x_curr_tensor)

                x_step = cp.Variable(n)
                # Maximize <grad_f, x> -> Minimize -<grad_f, x>
                prob_dca = cp.Problem(cp.Minimize(-grad_f_np @ x_step), get_constraints(x_step))
                prob_dca.solve(**SOLVER_OPTS)

                if x_step.value is None:
                    break
                x_curr = x_step.value
                if np.linalg.norm(x_curr - x_prev) < 1e-4:
                    break

            x_final_t = torch.tensor(x_curr, device=device, dtype=torch.float64)
            # Evaluate f(x) - g(x) (where g=0)
            final_obj = f_torch(x_final_t).item()

            if final_obj > best_val:
                best_val = final_obj
                best_x = x_curr

        return best_x



Running on device: cpu


# 2. PROBLEM DEFINITION

In [3]:

N = 300

# --- Functions (Torch) ---
def f11(x): return torch.sum(x)**2
def f21(x): return torch.sum((x**2))

def f_torch(x):
    return f11(x) + f21(x)

def f11_cp(x):

    return cp.square(cp.sum(x))

def f21_cp(x):

    return cp.sum_squares(x)


def f_cp(x):
    # Convex structure proxy for initialization
    return f11_cp(x) + f21_cp(x)

def g_torch(x): return torch.tensor(0.0, device=device)
def g_cp(x): return 0

# # --- Constraints ---
# # (1 + i)/90 <= xi <= (1 + 5i)/90
# indices = np.arange(1, N + 1)
# --- C. Constraints (UPDATED: -1/80 <= x <= 2/80) ---
val_low = -1.0 / 80.0
val_high = 2.0 / 80.0

# Create arrays filled with these values (Size N)
lower_bounds = np.full(N, val_low)
upper_bounds = np.full(N, val_high)

A1 = np.eye(N)
b1 = upper_bounds

A2 = -np.eye(N)
b2 = -lower_bounds

A_list = [A1, A2]
b_list = [b1, b2]
print(f"Problem: Maximize objective over {N} variables")
print(f"Constraint ranges: [{lower_bounds.min():.6f}, {upper_bounds.max():.6f}]")

# ==========================================
# 3. EXECUTION
# ==========================================
print("\nStarting MDCF solver...\n")

best_x = MDCF.solve(f_torch, g_torch, f_cp, g_cp, A_list, b_list, N)

# Report
sol_tensor = torch.tensor(best_x, device=device)
final_val = f_torch(sol_tensor).item()-g_torch(sol_tensor).item()

print("\n" + "="*50)
print(f"FINAL OBJECTIVE VALUE: {final_val:.6f}")

Problem: Maximize objective over 300 variables
Constraint ranges: [-0.012500, 0.025000]

Starting MDCF solver...

Step 1: Initialization...
Step 2: Computing analytic center...
  Analytic center computed. Mean = 0.006250
Step 3: Computing Hessian and gradient...
Step 4: Hidden convexity jump (x_in)...
Step 4b: Hidden convexity jump (x_out)...
Step 5: Line search...
Step 6: DCA refinement...
  Refining from candidate 1/3...
  Refining from candidate 2/3...
  Refining from candidate 3/3...

FINAL OBJECTIVE VALUE: 56.437501
